# Fridge measurements

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle

from scipy.optimize import curve_fit
from scipy.signal import find_peaks

import dexplore as dx 
import xarray as xr
import os
import sys
import glob
import re
from pathlib import Path

sys.path.append(os.path.abspath("steele-lab-analysis-functions"))
import analysis_functions.core as af
from analysis_functions.users.pacome._format_data_xarray import format_data_xarray

#from cavity_analysis import cavity_analysis
#from cavity_theory import cavity_modes

model = af.ComplexCircle()

In [ ]:
dx.refresh_bokeh()

In [ ]:
fc = 9.30154207e+09
fm = 1.21e+06 + 881

# VNA auto tuning

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/Automatic_tuning'

## Local Map ( GHz)

In [ ]:
file_folder = '2026-07-01_09.47.44_0001_automatic_tuning'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Cavity length (mm)':['length', 'mm', 1]}

local_map = af.format_data_xarray(data, tool='VNA', translator=translator)

fig, ax = plt.subplots()
local_map.mag.plot(ax=ax, cmap='RdBu_r')
fig.show()

## Last VNA trace

In [ ]:
file_folder = '2026-07-01_11.06.50_0013_automatic_tuning'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Cavity length (mm)':['length', 'mm', 1]}

last_vna = af.format_data_xarray(data, tool='VNA', translator=translator)

fig, ax = plt.subplots()
last_vna.mag.plot(ax=ax)
fig.show()

# SA auto tuning

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/SA_auto_tuning'

In [ ]:
file_folder = '2026-07-01_12.22.53_0001_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

data_background = format_data_xarray(data, tool='SA', translator=translator)

data_background.spect.plot()

In [ ]:
file_folder = '2026-07-01_12.23.39_0002_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

data_source = format_data_xarray(data, tool='SA', translator=translator)

data_source.spect.plot()
data

In [ ]:
file_folder = '2026-07-01_12.26.07_0003_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

data_cavity = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
data_cavity.spect.plot(ax = ax)
fig.show()


In [ ]:
fig, ax = plt.subplots()
data_source.spect.plot(ax = ax, color='blue', label='Source')
data_cavity.spect.plot(ax = ax, color='red', label='Cavity')
data_background.spect.plot(ax=ax, label='SA background', color='black', alpha=0.5)

ax.legend()
fig.show()

In [ ]:
attenuation = data_cavity.spect - data_source.spect

vna_rescale = last_vna.mag

vna_rescale.values = vna_rescale.values + (np.max(attenuation.values) - np.max(vna_rescale.values))

fig, ax = plt.subplots()
attenuation.plot(ax=ax)
last_vna.mag.plot(ax=ax, color='orange', label='VNA (offset)')
ax.axvline((fc - fm)*1e-9, color='red', linestyle='--', label='Pump')
ax.axvline(fc*1e-9, color='green', linestyle='--', label='Cavity')
ax.legend()
fig.show()

# Fridge (Off resonance)

In [ ]:
fc = 9.25 * 1e+09
fm = 1.21e+06 + 881

## VNA filter tuning

## SA Measurements

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/SA_Fridge_Off_Res'

### RBW = 5kHz

In [ ]:
file_folder = '2026-07-01_15.05.04_0001_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

background = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
background.spect.plot(ax = ax)
fig.show()

In [ ]:
file_folder = '2026-07-01_15.05.49_0002_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

source = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
source.spect.plot(ax = ax)
fig.show()

In [ ]:
file_folder = '2026-07-01_15.07.23_0003_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

cavity = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
cavity.spect.plot(ax = ax)
fig.show()

In [ ]:
fig, ax = plt.subplots()
source.spect.plot(ax=ax, color='blue', label='Source')
cavity.spect.plot(ax=ax, color='red', label='Cavity')
background.spect.plot(ax=ax, color='black', alpha=0.3, label='SA Background')
fig.show()

In [ ]:
attenuation = cavity.spect - source.spect

fig, ax = plt.subplots()
attenuation.plot(ax = ax)
fig.show()

### RBW = 10kHz

In [ ]:
file_folder = '2026-07-01_15.16.49_0004_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

background = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
background.spect.plot(ax = ax)
fig.show()

In [ ]:
file_folder = '2026-07-01_15.17.33_0005_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

source = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
source.spect.plot(ax = ax)
fig.show()

In [ ]:
file_folder = '2026-07-01_15.19.02_0006_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

cavity = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
cavity.spect.plot(ax = ax)
fig.show()

In [ ]:
fig, ax = plt.subplots()
source.spect.plot(ax=ax, color='blue', label='Source')
cavity.spect.plot(ax=ax, color='red', label='Cavity')
background.spect.plot(ax=ax, color='black', alpha=0.3, label='SA Background')
fig.show()

In [ ]:
attenuation = cavity.spect - source.spect

fig, ax = plt.subplots()
attenuation.plot(ax = ax)
fig.show()

### RBW = 10kHz, avg = 500

In [ ]:
file_folder = '2026-07-01_15.24.15_0007_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

background = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
background.spect.plot(ax = ax)
fig.show()

In [ ]:
file_folder = '2026-07-01_15.25.47_0008_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

source = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
source.spect.plot(ax = ax)
fig.show()

In [ ]:
file_folder = '2026-07-01_15.27.36_0009_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

cavity = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
cavity.spect.plot(ax = ax)
fig.show()

In [ ]:
fig, ax = plt.subplots()
source.spect.plot(ax=ax, color='blue', label='Source')
cavity.spect.plot(ax=ax, color='red', label='Cavity')
background.spect.plot(ax=ax, color='black', alpha=0.3, label='SA Background')
fig.show()

In [ ]:
attenuation = cavity.spect - source.spect

fig, ax = plt.subplots()
attenuation.plot(ax = ax)
fig.show()

# Cavity Map

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/Cavity_map'

## 4-8 GHz

In [ ]:
file_folder = '2026-07-01_16.11.58_0001_VNA_Cavity_Map'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Cavity length (mm)':['length', 'mm', 1]}

cavity_map = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
file_folder = '2026-07-02_10.41.28_0003_VNA_Cavity_Map'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Cavity length (mm)':['length', 'mm', 1]}

cavity_map = af.format_data_xarray(data, tool='VNA', translator=translator)

# Drums map (Matteo)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Drums'

In [ ]:
file_folder = '2026-07-01_16.20.58_0001_VNA_drums'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Cavity length (mm)':['length', 'mm', 1]}

drums_bf = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
file_folder = '2026-07-13_15.24.42_0036_VNA_drums'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Cavity length (mm)':['length', 'mm', 1]}

drums_af = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
drums_bf.isel(length=0).mag.plot(ax=ax, label='Previous Cooldown')
drums_af.isel(length=0).mag.plot(ax=ax, label='Current Cooldown')
ax.set_ylim(-20, -5)
ax.legend()
plt.show()

In [ ]:
file_folder = '2026-07-13_15.58.44_0038_VNA_drums'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Cavity length (mm)':['length', 'mm', 1]}

drums_af = af.format_data_xarray(data, tool='VNA', translator=translator)

# VNA auto tuning (Matteo)

## 6.860 GHz

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/Automatic_tuning_Matteo_Off_Res'

In [ ]:
file_folder = '2026-07-01_16.37.50_0001_automatic_tuning'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Cavity length (mm)':['length', 'mm', 1]}

tuning_map = af.format_data_xarray(data, tool='VNA', translator=translator)

## 6.800 GHz

In [ ]:
file_folder = '2026-07-01_17.00.45_0003_automatic_tuning'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Cavity length (mm)':['length', 'mm', 1]}

tuning_map = af.format_data_xarray(data, tool='VNA', translator=translator)

# Matteo Drum (6.845 GHz)

## Drum VNA Trace

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Drums'

In [ ]:
file_folder = '2026-07-02_11.26.13_0002_VNA_drums'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

drum_vna = af.format_data_xarray(data, tool='VNA', translator=translator)

## Power sweep

In [ ]:
file_folder = '2026-07-02_11.36.06_0004_VNA_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'power':['power', 'dBm', 1]}

drum_vna = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
file_folder = '2026-07-02_11.47.25_0005_VNA_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'power':['power', 'dBm', 1]}

drum_vna = af.format_data_xarray(data, tool='VNA', translator=translator)

drum_vna.mag.plot()

In [ ]:
peak_freqs = []

mag = drum_vna.mag

for i in range(mag.sizes["power"]):
    trace = mag.isel(power=i).values

    peaks, _ = find_peaks(-trace, prominence=1)

    freqs = np.full(2, np.nan)

    if len(peaks) > 0:
        # Trie les pics du plus profond au moins profond
        order = np.argsort(trace[peaks])
        peaks = peaks[order]

        n = min(2, len(peaks))
        freqs[:n] = mag.freq.values[peaks[:n]]

    peak_freqs.append(freqs)

peak_freqs = np.array(peak_freqs)

fig, ax = plt.subplots()

ax.plot(drum_vna.power, peak_freqs[:, 0], 'o', label='Cavity frequncy')
ax.plot(drum_vna.power, peak_freqs[:, 1], 'o', label='Blue Sideband')

ax.set_xlabel("Power [dBm]")
ax.set_ylabel("Frequency [GHz]")
ax.legend()

fig.tight_layout()
fig.show()

In [ ]:
delta_f = peak_freqs[:, 1] - peak_freqs[:, 0]

fig, ax = plt.subplots()

ax.plot(drum_vna.power, delta_f * 1e3, 'o--')

ax.set_xlabel("Power [dBm]")
ax.set_ylabel('Mechanical frequency [MHz]')
ax.grid(True)

plt.show()

## OMIT

In [ ]:
file_folder = '2026-07-02_12.26.13_0009_VNA_two_tone_pow'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

omit = af.format_data_xarray(data, tool='VNA', translator=translator)

**Cavity = 6.844957 GHz**

**Drum = 2.057970 MHz**

In [ ]:
6.844957e9-6.84289903e9

## Filter Off Resonance

### VNA Trace

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Filter'

In [ ]:
file_folder = '2026-07-02_13.44.51_0002_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

vna_filter = af.format_data_xarray(data, tool='VNA', translator=translator)

print(f"Filter = {vna_filter.freq.isel(freq = vna_filter.mag.argmin('freq')).values} GHz")

**Filter = 6.8627391 GHz**

### SA Measurements - Room Temperature

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/SA_Noise_Measurements'

In [ ]:
file_folder = '2026-07-02_13.52.03_0001_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

background = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
background.spect.plot(ax = ax)
fig.show()

In [ ]:
file_folder = '2026-07-02_13.54.40_0002_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

source = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
source.spect.plot(ax = ax)
fig.show()

In [ ]:
file_folder = '2026-07-02_13.56.14_0003_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

cavity = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
cavity.spect.plot(ax = ax)
fig.show()

In [ ]:
fig, ax = plt.subplots()
source.spect.plot(ax=ax, color='blue', label='Source')
cavity.spect.plot(ax=ax, color='red', label='Cavity')
background.spect.plot(ax=ax, color='black', alpha=0.3, label='SA Background')
fig.show()

In [ ]:
attenuation = cavity.spect - source.spect

fig, ax = plt.subplots()
attenuation.plot(ax = ax)
fig.show()

In [ ]:
f0 = source.freq.isel(freq = source.spect.argmax('freq'))

window = slice(f0 + 1e-4, source.freq.values[-1])
_source = source.sel(freq=window)
_cavity = cavity.sel(freq=window)

_source = _source.assign_coords(freq=(_source.freq - f0) * 1e9)

_cavity = _cavity.assign_coords(freq=(_cavity.freq - f0) * 1e9)

max_source = source.spect.max('freq')
max_cavity = cavity.spect.max('freq')
bandwidth = 10e+03

dBc_source = _source.spect - max_source - 10*np.log10(bandwidth)
dBc_cavity = _cavity.spect - max_cavity - 10*np.log10(bandwidth)

fig, ax = plt.subplots()
dBc_source.assign_coords(freq=_source.freq).plot(ax=ax, color='blue', label='Source')
dBc_cavity.assign_coords(freq=_cavity.freq).plot(ax=ax, color='red', label='Cavity')
ax.set_xscale('log')
ax.set_xlabel('Frequency shift [Hz]')
ax.set_ylabel('Noise [dBc/Hz]')
ax.legend()
fig.show()

### SA Measurements - Fridge

In [ ]:
file_folder = '2026-07-02_14.18.36_0004_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

background = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
background.spect.plot(ax = ax)
fig.show()

In [ ]:
file_folder = '2026-07-02_14.21.03_0005_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

source = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
source.spect.plot(ax = ax)
fig.show()

In [ ]:
file_folder = '2026-07-02_14.22.44_0006_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

cavity = format_data_xarray(data, tool='SA', translator=translator)

fig, ax = plt.subplots()
cavity.spect.plot(ax = ax)
fig.show()

In [ ]:
fig, ax = plt.subplots()
source.spect.plot(ax=ax, color='blue', label='Source')
cavity.spect.plot(ax=ax, color='red', label='Cavity')
background.spect.plot(ax=ax, color='black', alpha=0.3, label='SA Background')
fig.show()

In [ ]:
attenuation = cavity.spect - source.spect

fig, ax = plt.subplots()
attenuation.plot(ax = ax)
fig.show()

In [ ]:
f0 = source.freq.isel(freq = source.spect.argmax('freq'))

window = slice(f0 + 1e-4, source.freq.values[-1])
_source = source.sel(freq=window)
_cavity = cavity.sel(freq=window)

_source = _source.assign_coords(freq=(_source.freq - f0) * 1e9)

_cavity = _cavity.assign_coords(freq=(_cavity.freq - f0) * 1e9)

max_source = source.spect.max('freq')
max_cavity = cavity.spect.max('freq')
bandwidth = 10e+03

dBc_source = _source.spect - max_source - 10*np.log10(bandwidth)
dBc_cavity = _cavity.spect - max_cavity - 10*np.log10(bandwidth)

fig, ax = plt.subplots()
dBc_source.assign_coords(freq=_source.freq).plot(ax=ax, color='blue', label='Source')
dBc_cavity.assign_coords(freq=_cavity.freq).plot(ax=ax, color='red', label='Cavity')
ax.set_xscale('log')
ax.set_xlabel('Frequency shift [Hz]')
ax.set_ylabel('Noise [dBc/Hz]')
ax.legend()
fig.show()

## Filter tuning

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Filter'

In [ ]:
file_folder = '2026-07-02_15.17.22_0003_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

vna_filter = af.format_data_xarray(data, tool='VNA', translator=translator)

print(f"Filter = {vna_filter.freq.isel(freq = vna_filter.mag.argmin('freq')).values} GHz")

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Drums'

In [ ]:
file_folder = '2026-07-02_15.18.54_0017_VNA_drums'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

vna_filter = af.format_data_xarray(data, tool='VNA', translator=translator)
print(f"Cavity = {vna_filter.freq.isel(freq = vna_filter.mag.argmin('freq')).values} GHz")

## RSB Pump - Cavity SA measurements

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Drums'

### Test (we don't care)

In [ ]:
file_folder = '2026-07-02_13.30.07_0016_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
data = data.assign({
    "Spectrum (nW)": 1e9*data["Spectrum (W)"],
    "Spectrum (dBm)": 10*np.log10(data["Spectrum (W)"]) + 30,
})

d = list(data.dims)
dx.interactive_linecut_and_colormap(data)

### Without the filter

In [ ]:
file_folder = '2026-07-02_15.50.18_0023_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
data = data.assign({
    "Spectrum (nW)": 1e9*data["Spectrum (W)"],
    "Spectrum (dBm)": 10*np.log10(data["Spectrum (W)"]) + 30,
})

d = list(data.dims)
dx.interactive_linecut_and_colormap(data)

### With the filter

In [ ]:
file_folder = '2026-07-02_15.36.36_0022_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
data = data.assign({
    "Spectrum (nW)": 1e9*data["Spectrum (W)"],
    "Spectrum (dBm)": 10*np.log10(data["Spectrum (W)"]) + 30,
})

d = list(data.dims)
dx.interactive_linecut_and_colormap(data)

## RSB Pump - Cavity SA measurements (normalization of the source = 7.5dB)

### No Filter

In [ ]:
file_folder = '2026-07-02_17.36.26_0025_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
data = data.assign({
    "Spectrum (nW)": 1e9*data["Spectrum (W)"],
    "Spectrum (dBm)": 10*np.log10(data["Spectrum (W)"]) + 30,
})

d = list(data.dims)
dx.interactive_linecut_and_colormap(data)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

nofilter = format_data_xarray(data, tool='SA', translator=translator)

### Filter

In [ ]:
file_folder = '2026-07-02_17.20.56_0024_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
data = data.assign({
    "Spectrum (nW)": 1e9*data["Spectrum (W)"],
    "Spectrum (dBm)": 10*np.log10(data["Spectrum (W)"]) + 30,
})

d = list(data.dims)
dx.interactive_linecut_and_colormap(data)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

withfilter = format_data_xarray(data, tool='SA', translator=translator)

## RSB Pump - Cavity SA measurements (normalization of the source = 7.5dB)

### Filter Before / After (VNA Trace)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/Filter_tuning_Drum'

file_folder = '2026-07-03_09.42.58_0047_automatic_tuning'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Iteration (None)':['time', 'hour', 60]}

before = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Filter'

file_folder = '2026-07-03_10.49.37_0006_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Iteration (None)':['time', 'hour', 60]}

after = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
fig, ax = plt.subplots()
before.mag.plot(ax=ax, label='Before')
after.mag.plot(ax=ax, label='after')
ax.legend()
fig.show()

f0b = before.freq.isel(freq=before.mag.argmin('freq'))
f0a = after.freq.isel(freq=before.mag.argmin('freq'))
print(f'Frequency shift: {np.abs(f0b.values - f0a.values)*1e6:.3f} kHz')

### Background

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Drums'

In [ ]:
file_folder = '2026-07-03_10.45.11_0027_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
data = data.assign({
    "Spectrum (nW)": 1e9*data["Spectrum (W)"],
    "Spectrum (dBm)": 10*np.log10(data["Spectrum (W)"]) + 30,
})

d = list(data.dims)
dx.interactive_linecut_and_colormap(data)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

background = format_data_xarray(data, tool='SA', translator=translator)

### No filter (pump normalization = -5.577 dB)

In [ ]:
file_folder = '2026-07-03_10.57.38_0029_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
data = data.assign({
    "Spectrum (nW)": 1e9*data["Spectrum (W)"],
    "Spectrum (dBm)": 10*np.log10(data["Spectrum (W)"]) + 30,
})


d = list(data.dims)
dx.interactive_linecut_and_colormap(data)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

cooling_nf = format_data_xarray(data, tool='SA', translator=translator)

### With filter

In [ ]:
file_folder = '2026-07-03_09.48.01_0026_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
data = data.assign({
    "Spectrum (nW)": 1e9*data["Spectrum (W)"],
    "Spectrum (dBm)": 10*np.log10(data["Spectrum (W)"]) + 30,
})

d = list(data.dims)
dx.interactive_linecut_and_colormap(data)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

cooling_wf = format_data_xarray(data, tool='SA', translator=translator)
cooling_wf.spect.plot()

## Results

In [ ]:
def lorentzian(freq, f0, peak, gamma):
    return peak / (1 + (2 * (freq - f0) / gamma)**2)

def init_guess(freq, spect):
    f0 = freq[np.argmax(spect)]
    peak = np.max(spect)
    fwhm = peak / 2
    idx = np.where(spect >= fwhm)[0]
    gamma = freq[idx[-1]] - freq[idx[0]]
    
    return f0, peak, gamma

def init_guess(spectrum):
    f0 = spectrum.freq.isel(freq = spectrum.argmax('freq'))
    peak = spectrum.max('freq')
    fwhm = peak / 2
    
    mask = spectrum >= fwhm
    freq = spectrum.freq.where(mask)
    gamma = freq.max('freq') - freq.min('freq')

    return f0, peak, gamma


In [ ]:
def fit_lorentzian(spectrum):

    p0 = init_guess(spectrum)

    try:
        popt, pcov = curve_fit(lorentzian,
                               spectrum.freq,
                               spectrum,
                               p0=p0)
    except RuntimeError:
        popt = np.full(3, np.nan)
        pcov = np.full((3, 3), np.nan)

    return popt, pcov

In [ ]:
lorentz_nf = []
trapez_nf = []

for power in cooling_nf.power:
    spectrum = cooling_nf.sel(power=power, method='nearest').spect - background.spect
    popt, pcov = fit_lorentzian(spectrum)
    integral_lorentz = np.pi / 2 * popt[1] * popt[2]
    lorentz_nf.append(integral_lorentz)
    integral_trapez = np.trapz(spectrum, spectrum.freq)
    trapez_nf.append(integral_trapez)

    fig, ax = plt.subplots()
    spectrum.plot(ax=ax, color='blue', label='data')
    ax.plot(spectrum.freq, lorentzian(spectrum.freq, *popt), color='orange', label='fit')
    ax.legend()
    fig.show()


In [ ]:
cooling_wf = cooling_wf.assign_coords(
    power=((cooling_wf.power - 5.577))
)

In [ ]:
lorentz_wf = []
trapez_wf = []

for power in cooling_wf.power:
    spectrum = cooling_wf.sel(power=power, method='nearest').spect - background.spect
    popt, pcov = fit_lorentzian(spectrum)
    integral_lorentz = np.pi / 2 * popt[1] * popt[2]
    lorentz_wf.append(integral_lorentz)
    integral_trapez = np.trapz(spectrum, spectrum.freq)
    trapez_wf.append(integral_trapez)

    fig, ax = plt.subplots()
    spectrum.plot(ax=ax, color='blue', label='data')
    ax.plot(spectrum.freq, lorentzian(spectrum.freq, *popt), color='orange', label='fit')
    ax.legend()
    fig.show()


In [ ]:
power = cooling_nf.power

fig, ax = plt.subplots()
ax.plot(power, lorentz_nf, color='blue', marker='o', linestyle='--', label='Lorentzian')
ax.plot(power, trapez_nf, color='blue', marker='x', linestyle='--', label='Trapezoid')

ax.plot(power, lorentz_wf, color='red', marker='o', linestyle='--', label='Lorentzian')
ax.plot(power, trapez_wf, color='red', marker='x', linestyle='--', label='Trapezoid')

ax.legend()
ax.set_xlabel('Drive power [dBm]')
ax.set_ylabel('$S_{VV}$ [u.a.]')
fig.show()

# Power loss filter (for normalization of the pump)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Filter'

In [ ]:
file_folder = '2026-07-02_16.57.08_0004_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

vna_filter = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
file_folder = '2026-07-02_16.58.27_0005_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

vna_nofilter = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
attenuation = vna_filter.mag - vna_nofilter.mag

fig, ax = plt.subplots()
attenuation.plot(ax=ax)
fig.show()

In [ ]:
f_res = attenuation.isel(freq=attenuation.argmin('freq')).freq

attenuation_res = attenuation.sel(freq = f_res)
attenuation_pump = attenuation.sel(freq = f_res - 2.057970e-3, method='nearest')

print(f'Resonance: f={f_res.values:.6f} GHz | attenuation={attenuation_res.values:.3f} dB')
print(f'Pump: f={f_res.values - 2.057970e-3:.6f} GHz | attenuation={attenuation_pump.values:.3f} dB')

**Attenuation pump frequency = -7.5 dB**

Measurements with the SA with a pump at 0 dBm:

**Attenuation = -7.56 dB**

## With SA Measurements after fist power sweep

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/SA_Measurements_Source'

In [ ]:
file_folder = '2026-07-03_10.37.05_0001_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

withfilter = format_data_xarray(data, tool='SA', translator=translator)

withfilter.spect.max('freq')

In [ ]:
file_folder = '2026-07-03_10.38.53_0002_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

nofilter = format_data_xarray(data, tool='SA', translator=translator)

nofilter.spect.max('freq')

# Cooling measurements

## Cavity (VNA)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Drums'

In [ ]:
file_folder = '2026-07-03_12.03.49_0030_VNA_drums'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)
dx.interactive_linecut_and_colormap(data)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

cavity = af.format_data_xarray(data, tool='VNA', translator=translator)

## Filter evolution (VNA)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Filter'

In [ ]:
file_folder = '2026-07-03_12.03.22_0007_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

before = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
file_folder = '2026-07-03_12.03.22_0007_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

after = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
fig, ax = plt.subplots()
before.mag.plot(ax=ax, label='Before', color='blue')
after.mag.plot(ax=ax, label='After', color='red')

ax2 = ax.twinx()
cavity.mag.plot(ax=ax2, label='Cavity', color='green')
ax.legend()
ax2.legend(loc='lower right')
fig.show()

## Power normalization (SA)

### With filter (Before Thermal peaks)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/SA_Measurements_Source'

In [ ]:
file_folder = '2026-07-03_12.06.25_0003_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'power':['power', 'dBm', 1]}

before_30dB = format_data_xarray(data, tool='SA', translator=translator)

file_folder = '2026-07-03_12.06.45_0004_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'power':['power', 'dBm', 1]}

before_20dB = format_data_xarray(data, tool='SA', translator=translator)

file_folder = '2026-07-03_12.07.05_0005_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'power':['power', 'dBm', 1]}

before_0dB = format_data_xarray(data, tool='SA', translator=translator)

power = [-30, -20, 0]  # dBm

before = xr.concat(
    [before_30dB, before_20dB, before_0dB],
    dim=xr.DataArray(power, dims="power", name="power")
)
before.spect.plot()

### With filter (After Thermal peaks)

In [ ]:
file_folder = ''
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'power':['power', 'dBm', 1]}

after = format_data_xarray(data, tool='SA', translator=translator)

### Without filter

In [ ]:
file_folder = ''
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'power':['power', 'dBm', 1]}

nofilter = format_data_xarray(data, tool='SA', translator=translator)

### Pump normalization

In [ ]:
nofilter_max = nofilter.spect.isel(freq = after.spect.argmax('freq'))
after_max = after.spect.isel(freq = after.spect.argmax('freq'))

normalization = after_max - nofilter_max
normalization.plot()
print(normalization.mean('power').data)

## Thermal peaks (SA)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Drums'

### Background

In [ ]:
file_folder = '2026-07-03_10.45.11_0027_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
data = data.assign({
    "Spectrum (nW)": 1e9*data["Spectrum (W)"],
    "Spectrum (dBm)": 10*np.log10(data["Spectrum (W)"]) + 30,
})

d = list(data.dims)
dx.interactive_linecut_and_colormap(data)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

background = format_data_xarray(data, tool='SA', translator=translator)

### With cavity

In [ ]:
file_folder = '2026-07-03_12.13.58_0033_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
data = data.assign({
    "Spectrum (aW)": 1e18*data["Spectrum (W)"],
    "Spectrum (dBm)": 10*np.log10(data["Spectrum (W)"]) + 30,
})

d = list(data.dims)
dx.interactive_linecut_and_colormap(data)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

withfilter = format_data_xarray(data, tool='SA', translator=translator)

In [ ]:
def lorentzian(freq, f0, peak, gamma):
    return peak / (1 + (2 * (freq - f0) / gamma)**2)


def init_guess(spectrum):
    f0 = spectrum.freq.isel(freq = spectrum.argmax('freq'))
    peak = spectrum.max('freq')
    fwhm = peak / 2
    
    mask = spectrum >= fwhm
    freq = spectrum.freq.where(mask)
    gamma = freq.max('freq') - freq.min('freq')

    return f0, peak, gamma


In [ ]:
wf_remove_bg = withfilter.spect - background.spect
wf_remove_bg = wf_remove_bg.isel(power=slice(9, 19))
wf_power = wf_remove_bg.power

areas_wf = []

for p in wf_power.data:
    wf_spect = wf_remove_bg.sel(power=p)
    
    guess = init_guess(wf_spect.freq.data, wf_spect.data)
    
    popt, pcov = curve_fit(lorentzian, wf_spect.freq.data, wf_spect.data, p0=guess)
    
    f0, peak, gamma, = popt

    area = np.pi * peak * gamma / 2
    areas_wf.append(area)
    
    fit = lorentzian(test.freq.data, *popt)
    plt.plot(test.freq.data, fit)
    test.plot()
    plt.show()
    
fig, ax = plt.subplots()
ax.plot(wf_power, areas_wf, label='With filter')
ax.set_xlabel('Drive power [dBm]')
ax.set_ylabel('Lorentzian area [a.u.]')
fig.show()

### Without cavity

# Long time evolution

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/Long_Time_Evolution'

Start at 18h00. FAILED

In [ ]:
file_folder = '2026-07-01_17.48.26_0001_long_time_evolution'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Iteration (None)':['time', 'hour', 60]}

evolution = af.format_data_xarray(data, tool='VNA', translator=translator)

fig, ax = plt.subplots()
evolution.mag.plot(ax = ax)
fig.show()

In [ ]:
fmin = evolution.freq.isel(freq=evolution.isel(time=slice(0, 40)).mag.argmin("freq"))

delta_f = (fmin - fmin.isel(time=0))*1e6

fig, ax = plt.subplots()
delta_f.plot(ax=ax)
ax.set_ylabel('Frequency shift [kHz]')
fig.show()

Start at approx. 20h30

In [ ]:
file_folder = '2026-07-01_20.23.22_0002_long_time_evolution'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Iteration (None)':['time', 'hour', 60]}

evolution = af.format_data_xarray(data, tool='VNA', translator=translator)

fig, ax = plt.subplots()
evolution.mag.plot(ax = ax)
fig.show()

In [ ]:
fmin = evolution.freq.isel(freq=evolution.mag.argmin("freq"))

delta_f = (fmin - fmin.isel(time=0))*1e6

fig, ax = plt.subplots()
delta_f.plot(ax=ax)
ax.set_ylabel('Frequency shift [kHz]')
fig.show()

Started right after measure 0002

In [ ]:
file_folder = '2026-07-02_08.54.12_0003_long_time_evolution'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Iteration (None)':['time', 'hour', 60]}

evolution = af.format_data_xarray(data, tool='VNA', translator=translator)

fig, ax = plt.subplots()
evolution.mag.plot(ax = ax)
fig.show()

In [ ]:
fmin = evolution.freq.isel(freq=evolution.mag.argmin("freq"))

delta_f = (fmin - fmin.isel(time=0))*1e6

fig, ax = plt.subplots()
delta_f.plot(ax=ax)
ax.set_ylabel('Frequency shift [kHz]')
fig.show()

Started at 18h00, no AC, windows open

In [ ]:
file_folder = '2026-07-02_18.02.38_0004_long_time_evolution'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Iteration (None)':['time', 'hour', 60]}

evolution = af.format_data_xarray(data, tool='VNA', translator=translator,
                                 remove_edelay = [0, 1000, 0, 0])
evolution.mag.plot(cmap='plasma')

In [ ]:
fmin = evolution.freq.isel(freq=evolution.mag.argmin("freq"))

delta_f = (fmin - fmin.isel(time=0))*1e6

fig, ax = plt.subplots()
delta_f.plot(ax=ax)
ax.set_ylabel('Frequency shift [kHz]')
fig.show()

dBmin = evolution.mag.isel(freq=evolution.mag.argmin("freq"))

delta_dB = (dBmin - evolution.isel(freq=0).mag)

fig, ax = plt.subplots()
delta_dB.plot(ax=ax)
ax.set_ylabel('$|S_{21}(\omega=\omega_0)|$ [dB]')
fig.show()

In [ ]:
#evolution = evolution.isel(freq=slice(None, None, -1))
#evolution = af.bgd_remover_one_cut(evolution, window=[100,-100], order=1, is_plot=False)
fig, ax = plt.subplots(1,2,figsize=(10,4))
evolution.mag.plot(ax=ax[0], vmax=0, vmin=-90, cmap='plasma')
evolution.arg.plot(ax=ax[1])
fig.tight_layout()

In [ ]:
model = af.ComplexCircle() 
results = af.fit_many(model, evolution.freq.values, evolution.cpx.values)

In [ ]:
labels = {
    'f0': (r'$\omega_0/2\pi$', 'GHz'),
    'kt': (r'$\kappa_t/2\pi$', 'GHz'),
}
fit_many = af.many_results_to_xarray(results, evolution, model, labels=labels)
fit_many

In [ ]:
dBmin = evolution.mag.isel(freq=evolution.mag.argmin("freq"))

delta_dB = (dBmin - evolution.isel(freq=0).mag)

fit_many = fit_many.assign_coords(kt_MHz = fit_many.kt* 1e3)

fig, ax = plt.subplots(3,1,figsize=(9,6), sharex=True)
fit_many.f0.plot(ax=ax[0])
fit_many.kt_MHz.plot(ax=ax[1])
dBmin.plot(ax=ax[2])
ax[1].set_ylabel('$\kappa_t$ [MHz]')
ax[2].set_ylabel('$|S_{21}(\omega=\omega_0)|$ [dB]')
secay0 = ax[0].secondary_yaxis('right', functions=(lambda x: (x - fit_many.isel(time=0).f0.values)*1e6,
                                                  lambda x: x*1e-6 + fit_many.isel(time=0).f0.values))
secay0.set_ylabel("Shift [kHz]")
secay1 = ax[1].secondary_yaxis('right', functions=(lambda x: (x - fit_many.isel(time=0).kt_MHz.values)*1e3,
                                                  lambda x: x*1e-3 + fit_many.isel(time=0).kt_MHz.values))
secay1.set_ylabel("Shift [kHz]")
secay2 = ax[2].secondary_yaxis('right', functions=(lambda x: (x - dBmin.isel(time=0).values),
                                                  lambda x: x + dBmin.isel(time=0).values))
secay2.set_ylabel("Shift [dB]")
fig.tight_layout()

Started at 17h30, Friday 3 July

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/Long_Time_Evolution'

In [ ]:
file_folder = '2026-07-03_17.27.04_0005_long_time_evolution'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Iteration (None)':['time', 'hour', 30]}

evolution = af.format_data_xarray(data, tool='VNA', translator=translator,
                                 remove_edelay = [0, 1000, 0, 0])
fig, ax = plt.subplots()
evolution.mag.plot(ax=ax, cmap='plasma')
fig.show()

In [ ]:
fmin = evolution.freq.isel(freq=evolution.mag.argmin("freq"))

delta_f = (fmin - fmin.isel(time=0))*1e6

fig, ax = plt.subplots(figsize=(9, 6))
delta_f.plot(ax=ax, color='blue', label='Data')
ax.set_xlabel('Time [hour]')
ax.set_ylabel('Frequency shift [kHz]')
ax.axvspan(0, 2.5, color="gold", alpha=0.15, zorder=0, label='Day (6h - 20h)')
ax.axvspan(2.5, 12.5, color="steelblue", alpha=0.15, zorder=0, label='Night (20h - 6h)')
ax.axvspan(12.5, 26.5, color="gold", alpha=0.15, zorder=0)
ax.axvspan(26.5, 36.5, color="steelblue", alpha=0.15, zorder=0)
ax.axvspan(36.5, 50.5, color="gold", alpha=0.15, zorder=0)
ax.axvspan(50.5, 60.5, color="steelblue", alpha=0.15, zorder=0)
ax.axvspan(60.5, 63.5, color="gold", alpha=0.15, zorder=0)
ax.legend()
ax.set_title('Evolution of the filter frequency')
fig.show()

In [ ]:
model = af.ComplexCircle() 
results = af.fit_many(model, evolution.freq.values, evolution.cpx.values)

In [ ]:
labels = {
    'f0': (r'$\omega_0/2\pi$', 'GHz'),
    'kt': (r'$\kappa_t/2\pi$', 'GHz'),
}
fit_many = af.many_results_to_xarray(results, evolution, model, labels=labels)
fit_many

In [ ]:
dBmin = evolution.mag.isel(freq=evolution.mag.argmin("freq"))

delta_dB = (dBmin - evolution.isel(freq=0).mag)

fit_many = fit_many.assign_coords(kt_MHz = fit_many.kt* 1e3)

fig, ax = plt.subplots(3,1,figsize=(9,6), sharex=True)
fit_many.f0.plot(ax=ax[0])
fit_many.kt_MHz.plot(ax=ax[1])
dBmin.plot(ax=ax[2])
ax[1].set_ylabel('$\kappa_t$ [MHz]')
ax[2].set_ylabel('$|S_{21}(\omega=\omega_0)|$ [dB]')
secay0 = ax[0].secondary_yaxis('right', functions=(lambda x: (x - fit_many.isel(time=0).f0.values)*1e6,
                                                  lambda x: x*1e-6 + fit_many.isel(time=0).f0.values))
secay0.set_ylabel("Shift [kHz]")
secay1 = ax[1].secondary_yaxis('right', functions=(lambda x: (x - fit_many.isel(time=0).kt_MHz.values)*1e3,
                                                  lambda x: x*1e-3 + fit_many.isel(time=0).kt_MHz.values))
secay1.set_ylabel("Shift [kHz]")
secay2 = ax[2].secondary_yaxis('right', functions=(lambda x: (x - dBmin.isel(time=0).values),
                                                  lambda x: x + dBmin.isel(time=0).values))
secay2.set_ylabel("Shift [dB]")
fig.tight_layout()

# Long Time Optimization

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/Long_Time_Evolution'

Started at 20:30

New method: No circle fit, only minimum of the resonance to determine the resonance frequency

In [ ]:
file_folder = '2026-07-10_20.30.12_0020_long_time_optimization_ENA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Iteration (None)':['time', 'hour', 30]}

optimization = af.format_data_xarray(data, tool='VNA', translator=translator,
                                 remove_edelay = [0, 100, 0, 0])
fig, ax = plt.subplots()
optimization.mag.plot(ax=ax, cmap='plasma')
fig.show()

In [ ]:
fmin = optimization.freq.isel(freq=optimization.mag.argmin("freq"))

delta_f = (fmin - fmin.isel(time=0))*1e6

fig, ax = plt.subplots(figsize=(9, 6))
delta_f.plot(ax=ax, color='blue', label='Data')
ax.set_xlabel('Time [hour]')
ax.set_ylabel('Frequency shift [kHz]')

ax.axvspan(0, 0.001, color="gold", alpha=0.15, zorder=0, label='Day (6h - 20h)')
ax.axvspan(0.001, 9.5, color="steelblue", alpha=0.15, zorder=0, label='Night (20h - 6h)')
ax.axvspan(9.5, 23.5, color="gold", alpha=0.15, zorder=0)
ax.axvspan(23.5, 33.5, color="steelblue", alpha=0.15, zorder=0)
ax.axvspan(33.5, 47.5, color="gold", alpha=0.15, zorder=0)
ax.axvspan(47.5, 57.5, color="steelblue", alpha=0.15, zorder=0)
ax.axvspan(57.5, 60.5, color="gold", alpha=0.15, zorder=0)

ax.legend()
ax.set_title('Evolution of the filter frequency')
fig.show()

In [ ]:
fmin_ev = evolution.freq.isel(freq=evolution.mag.argmin("freq"))
fmin_opt = optimization.freq.isel(freq=optimization.mag.argmin("freq"))

delta_ev = (fmin_ev - fmin_ev.isel(time=0))*1e6
delta_opt = (fmin_opt - fmin_opt.isel(time=0))*1e6

fig, ax = plt.subplots(figsize=(9, 6))
delta_ev.plot(ax=ax, color='blue', label='Free evolution')
ax.plot(optimization.time + 3, delta_opt, color='red', label = 'Controled evolution')
#delta_opt.plot(ax=ax, color='red', label='Tuned evolution')
ax.set_xlabel('Time [hour]')
ax.set_ylabel('Frequency shift [kHz]')
ax.axvspan(0, 2.5, color="gold", alpha=0.15, zorder=0, label='Day (6h - 20h)')
ax.axvspan(2.5, 12.5, color="steelblue", alpha=0.15, zorder=0, label='Night (20h - 6h)')
ax.axvspan(12.5, 26.5, color="gold", alpha=0.15, zorder=0)
ax.axvspan(26.5, 36.5, color="steelblue", alpha=0.15, zorder=0)
ax.axvspan(36.5, 50.5, color="gold", alpha=0.15, zorder=0)
ax.axvspan(50.5, 60.5, color="steelblue", alpha=0.15, zorder=0)
ax.axvspan(60.5, 63.5, color="gold", alpha=0.15, zorder=0)
ax.legend()
ax.set_title('Evolution of the filter frequency')
fig.show()

# VNA Test

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_test'

file_folder = '2026-07-01_20.10.55_0007_VNA_freqsweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Iteration (None)':['time', 'hour', 60]}

evolution = af.format_data_xarray(data, tool='VNA', translator=translator)

fig, ax = plt.subplots()
evolution.mag.plot(ax = ax)
fig.show()

data

# Ringdown

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/Ringdown'

file_folder = '2026-07-03_13.45.55_0003_SA_IQ_time_ringdown'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

# translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
#               'Iteration (None)':['time', 'hour', 60]}

# evolution = af.format_data_xarray(data, tool='VNA', translator=translator)

# fig, ax = plt.subplots()
# evolution.mag.plot(ax = ax)
# fig.show()

data

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/Ringdown'

file_folder = '2026-07-03_13.50.22_0004_SA_IQ_time_ringdown'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)

# translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
#               'Iteration (None)':['time', 'hour', 60]}

# evolution = af.format_data_xarray(data, tool='VNA', translator=translator)

# fig, ax = plt.subplots()
# evolution.mag.plot(ax = ax)
# fig.show()

# data

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/Ringdown'

file_folder = '2026-07-03_14.04.52_0007_SA_IQ_time_ringdown'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut_and_colormap(data, initial_var=0)


# Two tones

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/ThermalNoiseTwoTones/'

In [ ]:
file_folder = '2026-07-03_14.31.11_0005_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

_data = format_data_xarray(data, tool='SA', translator=translator)
_data.spect.plot()


In [ ]:
file_folder = '2026-07-03_14.37.37_0006_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

_data = format_data_xarray(data, tool='SA', translator=translator)
_data.spect.plot()


In [ ]:
file_folder = '2026-07-03_14.42.24_0007_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

_data = format_data_xarray(data, tool='SA', translator=translator)
_data.spect.plot()


In [ ]:
file_folder = '2026-07-03_14.47.35_0009_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

_data = format_data_xarray(data, tool='SA', translator=translator)
_data.spect.plot()


In [ ]:
file_folder = '2026-07-03_14.53.44_0010_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

_data = format_data_xarray(data, tool='SA', translator=translator)
_data.spect.plot()

In [ ]:
file_folder = '2026-07-03_14.57.39_0011_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

_data = format_data_xarray(data, tool='SA', translator=translator)
_data.spect.plot()

In [ ]:
file_folder = '2026-07-03_15.01.51_0012_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

_data = format_data_xarray(data, tool='SA', translator=translator)
_data.spect.plot()

In [ ]:
file_folder = '2026-07-03_15.23.25_0018_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

_data = format_data_xarray(data, tool='SA', translator=translator)
_data.spect.plot()

In [ ]:
file_folder = '2026-07-03_15.26.51_0019_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

_data = format_data_xarray(data, tool='SA', translator=translator)
_data.spect.plot()

# Cooling

In [ ]:
f_cav = 6.844957e9
f_mech = 2.057970e6 + 75.6 - 66.20
print(f'Cavity = {f_cav*1e-9:.9f} GHz')
print(f'Mechamics = {f_mech*1e-6:.6f} MHz')

## VNA Measurements

### Cavity

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Drums/'

In [ ]:
file_folder = '2026-07-03_16.27.58_0034_VNA_drums'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

cavity = af.format_data_xarray(data, tool='VNA', translator=translator)

### Filter before

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Filter/'

In [ ]:
file_folder = '2026-07-03_16.28.17_0008_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

before = af.format_data_xarray(data, tool='VNA', translator=translator)

### Filter after

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Filter/'

In [ ]:
file_folder = '2026-07-03_17.11.00_0009_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

after = af.format_data_xarray(data, tool='VNA', translator=translator)

### Results

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset
from matplotlib.patches import Rectangle

f_cav = cavity.freq.isel(freq = cavity.mag.argmin('freq'))

fig, ax = plt.subplots(figsize=(10,6))
ax.plot((before.freq - f_cav)*1e3, before.mag, color='blue', label='Filter (before meas.)')
ax.plot((after.freq - f_cav)*1e3, after.mag, linestyle='--', color='blue', label='Filter (after meas.)')

ax.set_xlabel('Frequency shift [MHz]')
ax.set_ylabel('$|S_{21}|$ [dB]')
ax.set_xlim(-3, 5)

ax.spines['left'].set_color('blue')
ax.tick_params(axis='y', colors='blue')
ax.yaxis.label.set_color('blue')
ax.set_ylabel('$|S_{21}|$ [dB]')

ax2 = ax.twinx()
ax2.plot((cavity.freq - f_cav)*1e3, cavity.mag, color='red', label='Cavity')

ax2.spines['right'].set_color('red')
ax2.tick_params(axis='y', colors='red')
ax2.yaxis.label.set_color('red')
ax2.set_ylabel('$|S_{21}|$ [dB]')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax.legend(lines1 + lines2, labels1 + labels2, loc="lower left")

axins = ax.inset_axes([0.53, 0.13, 0.37, 0.37])

axins.plot((before.freq - f_cav)*1e6, before.mag, color='blue')
axins.plot((after.freq - f_cav)*1e6, after.mag, linestyle='--', color='blue')

axins.set_xlim(-100, 100)
axins.set_ylim(-47.7, -35)
axins.set_xlabel('Frequency shift [kHz]')
axins.set_ylabel('$|S_{21}|$ [dB]')

axins.spines['left'].set_color('blue')
axins.tick_params(axis='y', colors='blue')
axins.yaxis.label.set_color('blue')
axins.set_ylabel('$|S_{21}|$ [dB]')

rect = Rectangle((-0.1, -47),      # coin inférieur gauche
                 0.2,              # largeur (30 - 20)
                 13,            # hauteur (0.35 - 0.15)
                 fill=False,
                 linestyle='--',
                 edgecolor='black',
                 alpha=0.5,
                 linewidth=2)

ax.add_patch(rect)

ax3 = axins.twinx()
ax3.plot((cavity.freq - f_cav)*1e6, cavity.mag, color='red')
ax3.set_ylim(-16.3, -13.7)

ax3.spines['right'].set_color('red')
ax3.tick_params(axis='y', colors='red')
ax3.yaxis.label.set_color('red')
ax3.set_ylabel('$|S_{21}|$ [dB]')

ax.set_title('VNA Measurements of the cavity and the filter')

fig.tight_layout()
fig.show()

## Power normalization (SA measurements)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/SA_Measurements_Source/'

### Without filter

In [ ]:
file_folder = '2026-07-03_16.18.19_0007_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'power':['power', 'dBm', 1]}

nofilter = format_data_xarray(data, tool='SA', translator=translator)

nofilter.spect.plot()

### With filter

In [ ]:
file_folder = '2026-07-03_16.29.56_0008_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'power':['power', 'dBm', 1]}

withfilter = format_data_xarray(data, tool='SA', translator=translator)

withfilter.spect.plot()

### Normalization result

In [ ]:
nofilter_max = nofilter.spect.isel(freq = nofilter.spect.argmax('freq'))
withfilter_max = withfilter.spect.isel(freq = withfilter.spect.argmax('freq'))

normalization = withfilter_max - nofilter_max
normalization.plot()
normalization.mean('power').data

## Thermal peaks

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/ThermalNoiseTwoTones/'

### Background

In [ ]:
file_folder = '2026-07-03_15.38.08_0020_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

background = format_data_xarray(data, tool='SA', translator=translator)

### Without filter

In [ ]:
file_folder = '2026-07-03_15.41.14_0021_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

cooling_nf = format_data_xarray(data, tool='SA', translator=translator)
cooling_nf.spect.plot()

### With filter

In [ ]:
file_folder = '2026-07-03_16.35.19_0022_twotone_power_spectrum_SA'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'Drive power (dBm)':['power', 'dBm', 1]}

cooling_wf = format_data_xarray(data, tool='SA', translator=translator)
cooling_wf.spect.plot()

## Results

### Rescale dirve power

In [ ]:
cooling_wf = cooling_wf.assign_coords(
    power=((cooling_wf.power - 6.62167037))
)

### Lorentzian fit

In [ ]:
def lorentzian(freq, f0, peak, gamma):
    return peak / (1 + (2 * (freq - f0) / gamma)**2)

def init_guess(freq, spect):
    f0 = freq[np.argmax(spect)]
    peak = np.max(spect)
    fwhm = peak / 2
    idx = np.where(spect >= fwhm)[0]
    gamma = freq[idx[-1]] - freq[idx[0]]
    
    return f0, peak, gamma

def init_guess(spectrum):
    f0 = spectrum.freq.isel(freq = spectrum.argmax('freq'))
    peak = spectrum.max('freq')
    fwhm = peak / 2
    
    mask = spectrum >= fwhm
    freq = spectrum.freq.where(mask)
    gamma = freq.max('freq') - freq.min('freq')

    return f0, peak, gamma


In [ ]:
def fit_lorentzian(spectrum):

    p0 = init_guess(spectrum)

    try:
        popt, pcov = curve_fit(lorentzian,
                               spectrum.freq,
                               spectrum,
                               p0=p0)
    except RuntimeError:
        popt = np.full(3, np.nan)
        pcov = np.full((3, 3), np.nan)

    return popt, pcov

In [ ]:
lorentz_nf = []
trapez_nf = []

for power in cooling_nf.power:
    spectrum = cooling_nf.sel(power=power, method='nearest').spect - background.spect
    popt, pcov = fit_lorentzian(spectrum)
    integral_lorentz = np.pi / 2 * popt[1] * popt[2]
    lorentz_nf.append(integral_lorentz)
    integral_trapez = np.trapz(spectrum, spectrum.freq)
    trapez_nf.append(integral_trapez)
    
    fig, ax = plt.subplots()
    spectrum.plot(ax=ax, color='blue', label='data')
    ax.plot(spectrum.freq, lorentzian(spectrum.freq, *popt), color='orange', label='fit')
    ax.legend()
    fig.show()
    

In [ ]:
lorentz_wf = []
trapez_wf = []

for power in cooling_wf.power:
    spectrum = cooling_wf.sel(power=power, method='nearest').spect - background.spect
    popt, pcov = fit_lorentzian(spectrum)
    integral_lorentz = np.pi / 2 * popt[1] * popt[2]
    lorentz_wf.append(integral_lorentz)
    integral_trapez = np.trapz(spectrum, spectrum.freq)
    trapez_wf.append(integral_trapez)

    fig, ax = plt.subplots()
    spectrum.plot(ax=ax, color='blue', label='data')
    ax.plot(spectrum.freq, lorentzian(spectrum.freq, *popt), color='orange', label='fit')
    ax.legend()
    fig.show()


In [ ]:
power = cooling_nf.power
pol_nf = np.polyfit(power, trapez_nf, deg=1)
pol_wf = np.polyfit(power, trapez_wf, deg=1)

fig, ax = plt.subplots()
ax.plot(power, trapez_nf, color='blue', marker='x', label='Trapezoid')
ax.plot(power, np.polyval(pol_nf, power), color='blue', linestyle='--', alpha=0.5, label='Fit Trapezoid')

ax.plot(power, trapez_wf, color='red', marker='x', label='Trapezoid')
ax.plot(power, np.polyval(pol_wf, power), color='red', linestyle='--', alpha=0.5, label='Fit Trapezoid')
ax.legend()
fig.show()

In [ ]:
power = cooling_nf.power
pol_nf = np.polyfit(power, lorentz_nf, deg=1)
pol_wf = np.polyfit(power, lorentz_wf, deg=1)

fig, ax = plt.subplots()
ax.plot(power, lorentz_nf, color='blue', marker='x', label='Trapezoid')
ax.plot(power, np.polyval(pol_nf, power), color='blue', linestyle='--', alpha=0.5, label='Fit Trapezoid')

ax.plot(power, lorentz_wf, color='red', marker='x', label='Trapezoid')
ax.plot(power, np.polyval(pol_wf, power), color='red', linestyle='--', alpha=0.5, label='Fit Trapezoid')
ax.legend()
fig.show()

# Noise Sources Measurements

## RSB Pump

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/SA_Measurements_Source/'

In [ ]:
file_folder = '2026-07-06_10.27.01_0010_SA_measurements'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)
#dx.interactive_linecut_and_colormap(data, initial_var=0)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9],
              'power':['power', 'dBm', 1]}

cooling_nf = format_data_xarray(data, tool='SA', translator=translator)
cooling_nf.spect.plot()

## Probe

# IQ measurements

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/IQ_measurements/'

In [ ]:
file_folder = '2026-07-07_13.41.06_0004_IQ_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)

In [ ]:
file_folder = '2026-07-07_13.47.03_0005_IQ_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)

In [ ]:
file_folder = '2026-07-07_14.01.39_0006_IQ_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)

In [ ]:
file_folder = '2026-07-07_14.11.16_0007_IQ_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
dx.interactive_linecut_and_colormap(data)

In [ ]:
file_folder = '2026-07-07_14.37.37_0009_IQ_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)

In [ ]:
file_folder = '2026-07-07_14.48.20_0010_IQ_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)

In [ ]:
file_folder = '2026-07-08_08.46.12_0016_IQ_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)

In [ ]:
file_folder = '2026-07-08_08.47.55_0017_IQ_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)

In [ ]:
file_folder = '2026-07-08_08.50.03_0018_IQ_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)

In [ ]:
file_folder = '2026-07-08_08.52.14_0019_IQ_power_sweep'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=0)

# Noise cancellation (IQ Measurements & SOL calibration)

## SOL Calibration

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/SOL_Calibration/'

In [ ]:
file_folder = '2026-07-07_16.36.01_0001_VNA_SOLcal'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

short_cal = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
file_folder = '2026-07-07_16.39.03_0002_VNA_SOLcal'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

open_cal = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
file_folder = '2026-07-07_16.41.17_0003_VNA_SOLcal'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

load_cal = af.format_data_xarray(data, tool='VNA', translator=translator)

## VNA Trace (Filter)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Filter/'

In [ ]:
file_folder = '2026-07-07_17.56.25_0011_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

vna_trace = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
from scipy.interpolate import interp1d

def calculate_error_terms(short_cal, open_cal, load_cal):
    a_short = -1.0 + 0.0j
    a_open  =  1.0 + 0.0j
    
    # Directivity
    Ed = load_cal
    
    # Standard algebraic solution for the 3-term error model per frequency point
    denom = short_cal - open_cal
    # Avoid division by zero if data is bad
    denom[denom == 0] = 1e-15
    
    Es = ((short_cal - Ed) / a_short - (open_cal - Ed) / a_open) / denom
    Er = ((short_cal - Ed) * (1 - Es * a_short)) / a_short
    
    return Ed, Es, Er

def calibrate_mismatched_trace(freq_cal, short_cal, open_cal, load_cal, freq_trace, vna_trace):

    # 1. Calculate error terms at the calibration frequencies
    Ed_cal, Es_cal, Er_cal = calculate_error_terms(short_cal, open_cal, load_cal)
    
    # 2. Interpolate the complex error terms onto the DUT frequency grid
    # We interpolate the real and imaginary parts separately to preserve complex phase behavior
    Ed_interp = interp1d(freq_cal, Ed_cal.real, kind='cubic', fill_value="extrapolate")(freq_trace) + \
                1j * interp1d(freq_cal, Ed_cal.imag, kind='cubic', fill_value="extrapolate")(freq_trace)
                
    Es_interp = interp1d(freq_cal, Es_cal.real, kind='cubic', fill_value="extrapolate")(freq_trace) + \
                1j * interp1d(freq_cal, Es_cal.imag, kind='cubic', fill_value="extrapolate")(freq_trace)
                
    Er_interp = interp1d(freq_cal, Er_cal.real, kind='cubic', fill_value="extrapolate")(freq_trace) + \
                1j * interp1d(freq_cal, Er_cal.imag, kind='cubic', fill_value="extrapolate")(freq_trace)
    
    # 3. Apply the interpolated error terms to the DUT data
    calibrated_trace = (vna_trace - Ed_interp) / (Er_interp + Es_interp * (vna_trace - Ed_interp))
    
    return calibrated_trace

In [ ]:
calibrated_trace = calibrate_mismatched_trace(short_cal.freq,
                                              short_cal.re + 1j * short_cal.im,
                                              open_cal.re + 1j * open_cal.im,
                                              load_cal.re + 1j * load_cal.im,
                                              vna_trace.freq,
                                              vna_trace.re + 1j * vna_trace.im)

In [ ]:
fig, ax = plt.subplots()
ax.plot(vna_trace.freq, vna_trace.mag, color='blue', linestyle='-', label='Raw data')
ax.plot(vna_trace.freq, 20*np.log10(np.abs(calibrated_trace)), color='red', linestyle='-', label='Corrected data')
ax.set_xlabel('Frequency [GHz]')
ax.set_ylabel('$|S_{21}|^2$ [dB]')
ax.legend()
fig.show()

## IQ Measurements

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/IQ_measurements/'

In [ ]:
file_folder = '2026-07-07_17.27.22_0014_IQ_nofilter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

nofilter = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
dx.interactive_linecut_and_colormap(nofilter)
nofilter

In [ ]:
file_folder = '2026-07-07_17.41.04_0015_IQ_withfilter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

withfilter = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
dx.interactive_linecut_and_colormap(withfilter)

In [ ]:
freq = nofilter['freqs (Hz)']
PSD_nf = nofilter['PSD (V^2/Hz)'][:, 0]
PSD_wf = withfilter['PSD (V^2/Hz)'][:, 0]
Spectrum_nf = nofilter['Power spectrum (V^2)'][0]
Spectrum_wf = withfilter['Power spectrum (V^2)'][0]

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6, 9))
ax[0].semilogy(freq, PSD_nf, color='blue', label='Source')
ax[0].semilogy(freq, PSD_wf, color='red', label='Cavity')
ax[0].set_xlabel('Frequency [Hz]')
ax[0].set_ylabel('PSD [V^2/Hz]')
ax[0].legend()

vna_freq = vna_trace.freq * 1e9
vna_freq = vna_freq - 6.844957e9 +1e6

ax[1].semilogy(freq, PSD_wf / PSD_nf, color='green', label='$PSD_{filter}$ / $PSD_{source}$')
ax[1].semilogy(vna_freq, np.abs(calibrated_trace), color='orange', linestyle='--', label='Calibrated VNA trace')
ax[1].semilogy(vna_freq, np.abs(vna_trace.re + 1j*vna_trace.im), color='purple', linestyle='--', label='Raw VNA trace')
ax[1].set_xlabel('Frequency [Hz]')
ax[1].set_ylabel('Attenuation [u.a.]')
ax[1].legend()
fig.tight_layout()
fig.show()

## Same with more averages (and smaller acquisition time)

### SOL Calibration

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/SOL_Calibration/'

In [ ]:
file_folder = '2026-07-08_10.52.25_0004_VNA_SOLcal'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

short_cal = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
file_folder = '2026-07-08_10.53.31_0005_VNA_SOLcal'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

open_cal = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
file_folder = '2026-07-08_10.55.18_0006_VNA_SOLcal'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

load_cal = af.format_data_xarray(data, tool='VNA', translator=translator)

### VNA Traces (before and after)

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/VNA_Filter/'

In [ ]:
file_folder = '2026-07-08_08.58.11_0012_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

vna_before = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
file_folder = '2026-07-08_09.33.23_0013_VNA_filter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

data = xr.load_dataset(h5_files[0])
dx.interactive_linecut(data, initial_datavar=2)

translator = {'Frequency (Hz)':['freq', 'GHz', 1e9]}

vna_after = af.format_data_xarray(data, tool='VNA', translator=translator)

In [ ]:
calibrated_before = calibrate_mismatched_trace(short_cal.freq,
                                              short_cal.re + 1j * short_cal.im,
                                              open_cal.re + 1j * open_cal.im,
                                              load_cal.re + 1j * load_cal.im,
                                              vna_before.freq,
                                              vna_before.re + 1j * vna_before.im)

calibrated_after = calibrate_mismatched_trace(short_cal.freq,
                                              short_cal.re + 1j * short_cal.im,
                                              open_cal.re + 1j * open_cal.im,
                                              load_cal.re + 1j * load_cal.im,
                                              vna_after.freq,
                                              vna_after.re + 1j * vna_after.im)

In [ ]:
fig, ax = plt.subplots()
ax.plot(vna_before.freq, vna_before.mag, color='blue', linestyle='--', label='Before (Raw)')
ax.plot(vna_before.freq, 20*np.log10(np.abs(calibrated_before)), color='blue', linestyle='-', label='Before (Corrected)')
ax.plot(vna_after.freq, vna_after.mag, color='red', linestyle='--', label='After (Raw)')
ax.plot(vna_after.freq, 20*np.log10(np.abs(calibrated_after)), color='red', linestyle='-', label='After (Corrected)')
ax.set_xlabel('Frequency [GHz]')
ax.set_ylabel('$|S_{21}|^2$ [dB]')
ax.legend()
fig.show()

### IQ measurements

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/IQ_measurements/'

In [ ]:
file_folder = '2026-07-08_10.10.24_0022_IQ_background'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

background = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
dx.interactive_linecut_and_colormap(background)
background

In [ ]:
file_folder = '2026-07-08_09.35.24_0021_IQ_nofilter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

nofilter = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
dx.interactive_linecut_and_colormap(nofilter)

nofilter

In [ ]:
file_folder = '2026-07-08_09.00.10_0020_IQ_withfilter'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

withfilter = xr.load_dataset(h5_files[0])
#dx.interactive_linecut(data, initial_datavar=0)
dx.interactive_linecut_and_colormap(withfilter)
withfilter

In [ ]:
freq = nofilter['freqs (Hz)']
PSD_bg = background['PSD (V^2/Hz)'][:, 0]
PSD_nf = nofilter['PSD (V^2/Hz)'][:, 0]
PSD_wf = withfilter['PSD (V^2/Hz)'][:, 0]

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6, 9))
ax[0].semilogy(freq, PSD_bg, color='black', alpha=0.5, label='SA background')
ax[0].semilogy(freq, PSD_nf, color='blue', label='Source')
ax[0].semilogy(freq, PSD_wf, color='red', label='Cavity')
ax[0].set_xlabel('Frequency [Hz]')
ax[0].set_ylabel('PSD [V^2/Hz]')
ax[0].legend()

vna_freq = vna_before.freq * 1e9
vna_freq = vna_freq - 6.844957e9 +1e6

ax[1].semilogy(freq, PSD_wf / PSD_nf, color='green', label='$PSD_{filter}$ / $PSD_{source}$')
ax[1].semilogy(vna_freq, np.abs(calibrated_before), color='orange', linestyle='--', label='Calibrated VNA (Before PSD meas.)')
ax[1].semilogy(vna_freq, np.abs(calibrated_after), color='purple', linestyle='--', label='Calibrated VNA (After PSD meas.)')
ax[1].set_xlabel('Frequency [Hz]')
ax[1].set_ylabel('Attenuation [u.a.]')
ax[1].legend()
fig.tight_layout()
fig.show()

## Background stuff (50$\Omega$ termination on the SA)

In [ ]:
f_cav = 6.844957e9
f_mech = 2.057970e6 + 75.6 - 66.20

In [ ]:
base_folder = '/home/jovyan/steelelab/measurement_data/newBF/Pacome/IQ_measurements/'

f_center = f_cav - f_mech + 1MHz = 6.843899 GHz

In [ ]:
file_folder = '2026-07-08_10.57.22_0023_IQ_background'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

background = xr.load_dataset(h5_files[0])
dx.interactive_linecut(background, initial_datavar=1)
#dx.interactive_linecut_and_colormap(background)

f_center = f_cav - f_mech + 0MHz = 6.842899 GHz

In [ ]:
file_folder = '2026-07-08_11.38.29_0024_IQ_background'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

background = xr.load_dataset(h5_files[0])
dx.interactive_linecut(background, initial_datavar=1)
#dx.interactive_linecut_and_colormap(background)

In [ ]:
file_folder = '2026-07-08_12.13.13_0025_IQ_background'
full_path = base_folder + '/' + file_folder
h5_files = sorted(glob.glob(full_path + '/*.h5'))

background = xr.load_dataset(h5_files[0])
dx.interactive_linecut(background, initial_datavar=1)
#dx.interactive_linecut_and_colormap(background)